# Myfxbook EIA consensus — preprocessing

Этот ноутбук **не скачивает данные из Myfxbook**.

Он берет уже сохраненный raw JSON:

`data/raw/consensus/myfxbook_eia_crude_oil_stocks_12y.json`

и приводит его к формату, пригодному для объединения с недельными фундаментальными данными EIA.

На выходе создаются:

- `data/processed/consensus/myfxbook_eia_consensus.parquet`
- `data/processed/consensus/myfxbook_eia_consensus.csv`

Ключевые поля:

- `release_datetime` — момент публикации;
- `release_date` — дата публикации;
- `eia_week_ending` — пятница, которой соответствует отчетная неделя EIA;
- `actual_mmbbl` — фактическое изменение запасов, млн баррелей;
- `consensus_mmbbl` — консенсус-прогноз, млн баррелей;
- `previous_mmbbl` — предыдущее значение, млн баррелей;
- `inventory_surprise_mmbbl = actual_mmbbl - consensus_mmbbl`.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


In [2]:
# Находим корень репозитория.
repo_root = Path.cwd().resolve()

while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

if not (repo_root / "pyproject.toml").exists():
    raise RuntimeError("Не удалось найти корень репозитория с pyproject.toml")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root


PosixPath('/Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph')

In [3]:
from src.data.myfxbook_consensus import normalize_myfxbook_payload


In [4]:
RAW_PATH = (
    repo_root
    / "data"
    / "external"
    / "myfxbook"
    / "myfxbook_eia_crude_oil_stocks_12y.json"
)

PROCESSED_DIR = repo_root / "data" / "processed" / "consensus"
PARQUET_PATH = PROCESSED_DIR / "myfxbook_eia_consensus.parquet"
CSV_PATH = PROCESSED_DIR / "myfxbook_eia_consensus.csv"

if not RAW_PATH.exists():
    raise FileNotFoundError(
        "Raw-файл Myfxbook не найден. "
        f"Ожидаемый путь: {RAW_PATH}"
    )

print(f"Используем raw-файл: {RAW_PATH}")
print(f"Размер: {RAW_PATH.stat().st_size / 1024:.1f} KB")


Используем raw-файл: /Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/external/myfxbook/myfxbook_eia_crude_oil_stocks_12y.json
Размер: 21.6 KB


In [5]:
with RAW_PATH.open("r", encoding="utf-8") as file:
    payload = json.load(file)

print(type(payload))


<class 'dict'>


## Приведение к единому формату

Фильтруем период проекта, начиная с 2015 года. Верхнюю границу специально не задаем, чтобы при обновлении raw-файла новые релизы автоматически попадали в обработанный датасет.


In [6]:
categories = payload["categories"]

series_by_name = {
    item["name"]: item["data"]
    for item in payload["series"]
}

actual = series_by_name["Actual"]
consensus_values = series_by_name["Consensus"]

assert len(categories) == len(actual)
assert len(categories) == len(consensus_values)

consensus = pd.DataFrame(
    {
        "release_date": pd.to_datetime(
            categories,
            format="%b %d, '%y",
        ),
        "actual_bbl": actual,
        "consensus_bbl": consensus_values,
    }
)

consensus["actual_mmbbl"] = (
    pd.to_numeric(
        consensus["actual_bbl"],
        errors="coerce",
    )
    / 1_000_000
)

consensus["consensus_mmbbl"] = (
    pd.to_numeric(
        consensus["consensus_bbl"],
        errors="coerce",
    )
    / 1_000_000
)

consensus["inventory_surprise_mmbbl"] = (
    consensus["actual_mmbbl"]
    - consensus["consensus_mmbbl"]
)

consensus = (
    consensus[
        consensus["release_date"] >= "2015-01-01"
    ]
    .reset_index(drop=True)
)

consensus.head()

,release_date,actual_bbl,consensus_bbl,actual_mmbbl,consensus_mmbbl,inventory_surprise_mmbbl
0,2015-01-07,-3062000.0,900000.0,-3.062,0.90,-3.962
1,2015-01-14,5400000.0,400000.0,5.400,0.40,5.000
2,2015-01-22,10070000.0,2600000.0,10.070,2.60,7.470
3,2015-01-28,8874000.0,3800000.0,8.874,3.80,5.074
4,2015-02-04,6333000.0,3750000.0,6.333,3.75,2.583


In [7]:
consensus.tail()


,release_date,actual_bbl,consensus_bbl,actual_mmbbl,consensus_mmbbl,inventory_surprise_mmbbl
613,2026-10-07,-3186000.0,1700000.0,-3.186,1.7,-4.886
614,2026-10-15,-8557000.0,-300000.0,-8.557,-0.3,-8.257
615,2026-10-21,-3061000.0,-800000.0,-3.061,-0.8,-2.261
616,2026-10-28,-1892000.0,0.0,-1.892,0.0,-1.892
617,2026-11-04,-865000.0,900000.0,-0.865,0.9,-1.765


## Оставляем релизы, для которых есть consensus

Для модели нас интересует именно ошибка ожиданий рынка:

`inventory_surprise_mmbbl = actual_mmbbl - consensus_mmbbl`

Поэтому строки без `consensus_mmbbl` не могут использоваться как consensus-feature.


In [8]:
print("Строк до фильтрации:", len(consensus))
print("Пропусков consensus:", consensus["consensus_mmbbl"].isna().sum())
print("Пропусков actual:", consensus["actual_mmbbl"].isna().sum())

consensus = (
    consensus
    .dropna(subset=["consensus_mmbbl"])
    .sort_values("release_date")
    .drop_duplicates(subset=["release_date"], keep="last")
    .reset_index(drop=True)
)

print("Строк после фильтрации:", len(consensus))


Строк до фильтрации: 618
Пропусков consensus: 1
Пропусков actual: 0
Строк после фильтрации: 616


## Сохранение

Parquet — основной формат для дальнейшего pipeline.  
CSV оставляем для быстрой ручной проверки.


In [9]:
consensus.to_parquet(
    PARQUET_PATH,
    index=False,
)

consensus.to_csv(
    CSV_PATH,
    index=False,
)

print(f"Parquet: {PARQUET_PATH}")
print(f"CSV:     {CSV_PATH}")


Parquet: /Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/processed/consensus/myfxbook_eia_consensus.parquet
CSV:     /Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/processed/consensus/myfxbook_eia_consensus.csv


In [10]:
# Проверяем, что сохраненный parquet читается без проблем.
check = pd.read_parquet(PARQUET_PATH)

print(check.shape)
check.head()


(616, 6)


,release_date,actual_bbl,consensus_bbl,actual_mmbbl,consensus_mmbbl,inventory_surprise_mmbbl
0,2015-01-07,-3062000.0,900000.0,-3.062,0.90,-3.962
1,2015-01-14,5400000.0,400000.0,5.400,0.40,5.000
2,2015-01-22,10070000.0,2600000.0,10.070,2.60,7.470
3,2015-01-28,8874000.0,3800000.0,8.874,3.80,5.074
4,2015-02-04,6333000.0,3750000.0,6.333,3.75,2.583
